In [59]:
import torch
from torch.jit import script , trace
import torch.nn as nn
from torch import optim
import torch.nn.functional as F
import csv
import re 
import os
import unicodedata
import codecs
from io import open
import itertools
import math
import json
import random

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu'
print(f"Using {device} device")

Using cpu device


# Load & Preprocess data

In [60]:
corpus_name = "movie-corpus"
corpus = os.path.join('data', corpus_name)

def printlines(file,n=10):
    with open(file ,'rb') as f:
        lines = f.readlines()
    for l in lines[:n]:
        print(l)

printlines(os.path.join(corpus,'utterances.jsonl'))

b'{"id": "L1045", "conversation_id": "L1044", "text": "They do not!", "speaker": "u0", "meta": {"movie_id": "m0", "parsed": [{"rt": 1, "toks": [{"tok": "They", "tag": "PRP", "dep": "nsubj", "up": 1, "dn": []}, {"tok": "do", "tag": "VBP", "dep": "ROOT", "dn": [0, 2, 3]}, {"tok": "not", "tag": "RB", "dep": "neg", "up": 1, "dn": []}, {"tok": "!", "tag": ".", "dep": "punct", "up": 1, "dn": []}]}]}, "reply-to": "L1044", "timestamp": null, "vectors": []}\n'
b'{"id": "L1044", "conversation_id": "L1044", "text": "They do to!", "speaker": "u2", "meta": {"movie_id": "m0", "parsed": [{"rt": 1, "toks": [{"tok": "They", "tag": "PRP", "dep": "nsubj", "up": 1, "dn": []}, {"tok": "do", "tag": "VBP", "dep": "ROOT", "dn": [0, 2, 3]}, {"tok": "to", "tag": "TO", "dep": "dobj", "up": 1, "dn": []}, {"tok": "!", "tag": ".", "dep": "punct", "up": 1, "dn": []}]}]}, "reply-to": null, "timestamp": null, "vectors": []}\n'
b'{"id": "L985", "conversation_id": "L984", "text": "I hope so.", "speaker": "u0", "meta": {

In [61]:
# Splits each line of the file to create lines and conversations

def loadlinesAndConversations(filename):
    lines={}
    conversations ={}
    with open (filename, 'r', encoding='iso-8859-1') as f:
        for line in f:
            lineJson = json.loads(line)
            # Extract fields for line object
            lineObj= {}
            lineObj['lineID']  = lineJson['id']
            lineObj['characterID'] = lineJson['speaker']
            lineObj['text'] = lineJson['text']
            lines[lineObj['lineID']] = lineObj

            # extract fields for conversation object
            if lineJson['conversation_id'] not in conversations:
                convObj = {}
                convObj['conversationID'] =lineJson['conversation_id']
                convObj['movieID'] =lineJson['meta']['movie_id']
                convObj['lines'] =[lineObj]
            
            else:
                convObj= conversations[lineJson['conversation_id']]
                convObj['lines'].insert(0,lineObj)
            conversations[convObj['conversationID']] = convObj
    return lines, conversations


#Extract pairs of sentwnces from conversations
def extractSentencePairs(conversations):
    qa_pairs =[]
    for conversation in conversations.values():
        for i in range(len(conversation['lines'])- 1):
            inputLine = conversation['lines'][i]['text'].strip()
            targetine = conversation['lines'][i+1]['text'].strip()
            if inputLine and targetine:
                qa_pairs.append([inputLine, targetine])
    return qa_pairs


In [62]:
datafile = os.path.join(corpus, 'formatted_movie_lines.txt')

delimiter= '\t'
delimiter= str(codecs.decode(delimiter,'unicode_escape'))
lines ={}
conversations={}
print("\nProcessing corpus into lines and conversations....")
lines, conversations =loadlinesAndConversations(os.path.join(corpus,"utterances.jsonl"))

print("\nWriting newly formatted file...")
with open(datafile,'w',encoding='utf-8') as outputfile:
    writer= csv.writer(outputfile,delimiter=delimiter,lineterminator='\n')
    for pair in extractSentencePairs(conversations):
        writer.writerow(pair)

print('\nSampple lines for line:')
printlines(datafile)



Processing corpus into lines and conversations....

Writing newly formatted file...

Sampple lines for line:
b'They do to!\tThey do not!\n'
b'She okay?\tI hope so.\n'
b"Wow\tLet's go.\n"
b'"I\'m kidding.  You know how sometimes you just become this ""persona""?  And you don\'t know how to quit?"\tNo\n'
b"No\tOkay -- you're gonna need to learn how to lie.\n"
b"I figured you'd get to the good stuff eventually.\tWhat good stuff?\n"
b'What good stuff?\t"The ""real you""."\n'
b'"The ""real you""."\tLike my fear of wearing pastels?\n'
b'do you listen to this crap?\tWhat crap?\n'
b"What crap?\tMe.  This endless ...blonde babble. I'm like, boring myself.\n"


# Load and trim Data

In [63]:
PAD_token =0
SOS_token= 1
EOS_token =2

class Voc:
    def __init__(self,name):
        self.name = name 
        self.trimmed =False
        self.word2index = {}
        self.word2count={}
        self.index2owrd={PAD_token:'PAD', SOS_token:"SOS", EOS_token:"EOS"}
        self.num_words =3

    def addsentence(self, sentence):
        for word in sentence.split(' '):
            self.addword(word)
    
    def addword(self,word):
        if word not in self.word2index:
            self.word2index[word] =self.num_words
            self.word2count[word] = 1
            self.index2owrd[self.num_words] =word
            self.num_words +=1
        else:
            self.word2count[word] +=1
    
    # remove words below a certain count threshold
    def trim (self, min_count):
        if self.trimmed:
            return
        self.trimmed=True
        keep_words = []
        for k,v in  self.word2count.items():
            if v>= min_count:
                keep_words.append(k)

        print('keep_words {} / {} = {:.4f}'.format(
            len(keep_words), len(self.word2index), len(keep_words)/ len(self.word2index)
        ))
        self.word2index={}
        self.word2count={}
        self.index2owrd={PAD_token:"PAD", SOS_token:"SOS", EOS_token:"EOS"}
        self.num_words =3

        for word in keep_words:
            self.addword(word)

In [ ]:
MAX_LENGTH =10
def unicodetoassci(s):
    return "".join(
        c for c in unicodedata.normalize('NFD',s)
        if unicodedata.category(c) != 'Mn'
    )

def normalizestring(s):
    s = unicodetoassci(s.lower().strip())
    s= re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z.!?]+", r" ",s)
    r = re.sub(r"\s+", r" ",s).strip()

    return s

def readvocs(datafile ,corpus_name):
    print("Reading lines ...")
    lines = open(datafile, encoding='utf-8').\
        read().strip().split('\n')

    pairs = [[normalizestring(s) for s in l.split('\t')] for l in lines]
    voc = Voc(corpus_name)
    return voc , pairs

def filterpair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and len(p[1].split(' ')) <MAX_LENGTH

def filterspairs(pairs):
    return [pair for pair in pairs if filterpair(pair)]


def loadprepapredata(corpus, corpus_name, datafile, save_dir):
    print("start preparing trainging data...")
    voc, pairs = readvocs(datafile, corpus_name)
    print(f"Read {len(pairs)} sesntence pairs")
    pairs= filterspairs(pairs)
    print("Trimmed to {!s} sentense pairs".format(len(pairs)))
    print("COunting words....")
    for pair in pairs:
        voc.addsentence(pair[0])
        voc.addsentence(pair[1])
    print('COunted words:', voc.num_words)
    return voc, pairs 


save_dir = os.path.join("data", 'save')
voc, pairs = loadprepapredata(corpus, corpus_name, datafile, save_dir)
print('\npairs')
for pair in pairs[:10]:
    print(pair)

start preparing trainging data...
Reading lines ...


In [ ]:
MIN_COUNT = 3
def trimrarewords(voc, pairs, MIN_COUNT):
    voc.trim(MIN_COUNT)
    keep_pairs = []
    for pair in pairs:
        input_sentence =pair[0]
        output_sentence = pair[1]
        keep_input = True
        keep_output = True

        for word in input_sentence.split(' '):
            if word not in voc.word2index:
                keep_input= False
                break
        for word  in output_sentence.split(" "):
            if word not in voc.word2index:
                keep_output= False
                break
        if keep_input and keep_output:
            keep_pairs.append(pair)

    print(f'Trimmed form {len(pairs)} pairs to {len(keep_pairs)}, {len(keep_pairs)/len(pairs)} of total')
    return keep_pairs


pairs = trimrarewords(voc, pairs, MIN_COUNT)


keep_words 7716 / 17844 = 0.4324
Trimmed form 63499 pairs to 52433, 0.8257295390478591 of total


# Prepapre data for models

In [ ]:
def indexesfromsentence(voc, sentence):
    return [voc.word2index[word] for word in sentence.split(" ")]+ [EOS_token]

def zeropadding(l, fillvalue=PAD_token):
    return list(itertools.zip_longest(*l, fillvalue=fillvalue))

def binarymatrix(l, value=PAD_token):
    m=[]
    for i ,seq in enumerate(l):
        m.append([])
        for token in seq:
            if token == PAD_token:
                m[i].append(0)
            else:
                m[i].append(1)
    return m

def inputvar(l,voc):
    indexes_batch = [indexesfromsentence(voc,sentence) for sentence in l]
    lengths = torch.tensor([len(indexes) for indexes in indexes_batch])
    padlist= zeropadding(indexes_batch)
    padvar= torch.LongTensor(padlist)
    return padvar, lengths

def outputvar(l, voc):
    index_batch = [indexesfromsentence(voc, sentence) for sentence in l]
    max_target_len = max([len(indexes) for indexes in index_batch])
    padlist= zeropadding(index_batch)
    mask = binarymatrix(padlist)
    mask = torch.BoolTensor(mask)
    padvar = torch.LongTensor(padlist)
    return padvar,mask, max_target_len

def batch2traindata(voc, pair_batch):
    pair_batch.sort(key =lambda x: len(x[0].split(" ")), reverse= True)
    inputbatch, outputbatch = [],[]
    for pair in pair_batch:
        inputbatch.append(pair[0])
        outputbatch.append(pair[1])

    inp,lenghts = inputvar(inputbatch, voc)
    output, mask,max_target_len = outputvar(outputbatch, voc)
    return inp, lenghts, output,mask, max_target_len

small_batch =5
batches = batch2traindata(voc, [random.choice(pairs) for _ in range(small_batch)])
inputvariable, lenghts, target_variable , mask, max_target_len = batches

print("input variable: ", inputvariable)
print("lenghts: ", lenghts)
print("target variables:" ,target_variable)
print("mask:", mask)
print("max_target_len: ", max_target_len)


input variable:  tensor([[  11, 1580,   23,   19,   19],
        [ 187,   17, 1171,  177,   17],
        [  86,   20,  119,   25,   93],
        [1132,  510, 1579,    4,   10],
        [  14,   90,   14,   10,    2],
        [  11,   14,    2,    2,    0],
        [ 187,    2,    0,    0,    0],
        [  86,    0,    0,    0,    0],
        [  14,    0,    0,    0,    0],
        [   2,    0,    0,    0,    0]])
lenghts:  tensor([10,  7,  6,  6,  5])
target variables: tensor([[  19,  105,   37,   11,   34],
        [   4,  119,   13,  504,   11],
        [  25,   63,   10,  142,  201],
        [ 187,   69,    2,   14,  475],
        [  10, 1432,    0,    2,   14],
        [   2,    2,    0,    0,  127],
        [   0,    0,    0,    0,   14],
        [   0,    0,    0,    0,   14],
        [   0,    0,    0,    0,   14],
        [   0,    0,    0,    0,    2]])
mask: tensor([[ True,  True,  True,  True,  True],
        [ True,  True,  True,  True,  True],
        [ True,  True,  True

# Define model
 Seq2Seq

In [ ]:
class EncoderRNN(nn.Module):
    def __init__(self,hidden_size, embedding ,n_layers=1, dropout=0):
        super().__init__()
        self.n_layers= n_layers
        self.hidden_size = hidden_size
        self.embedding = embedding
      
        self.gru = nn.GRU(hidden_size,hidden_size,n_layers,
                          dropout=(0 if n_layers == 1 else dropout), bidirectional=True)

    def forward(self, input_seq, input_lengths, hidden=None):
        # convert word indexes to embeddings
        embedded = self.embedding(input_seq)
        #pack padded batch of sequences for RNN module
        packed = nn.utils.rnn.pack_padded_sequence(embedded, input_lengths)
        # forward pass through GRU
        outputs, hidden = self.gru(packed, hidden)
        # Unpack padding 
        outputs, _ = nn.utils.rnn.pad_packed_sequence(outputs)
        # sum bidirectional GRU outputs
        outputs = outputs[:, :, :self.hidden_size] + outputs[:, : ,self.hidden_size:]
        
        # return output and final hidden state
        return outputs, hidden

In [ ]:
# Luong attention layer
class Attn(nn.Module):
    def __init__(self, method, hidden_size):
        super().__init__()
        self.method = method
        if self.method not in ['dot', 'general','concat']:
            raise ValueError(self.method, "is not an appropriate attention method")
        self.hidden_size = hidden_size
        if self.method =='general':
            self.attn= nn.Linear(self.hidden_size, hidden_size)
        elif self.method =='concat':
            self.attn = nn.Linear(self.hidden_size*2,hidden_size)
            self.v =nn.Parameter(torch.FloatTensor(hidden_size))

    def dot_score(self, hidden, encoder_output):
        return torch.sum(hidden*encoder_output, dim=2)
     
    def general_score (self,hidden,encoder_output):
        energy =self.attn(encoder_output)
        return torch.sum(hidden*energy, dim=2)  

    def concat_score(self, hidden, encoder_output):
        energy = self.attn(torch.cat((hidden.expand(encoder_output.size(0), -1 ,-1), encoder_output),2)).tanh()
        return torch.sum(self.v* energy,dim=2)
    
    def forward(self,hidden, encoder_output):
        # calculate the attention weights (energies) based on the given methed
        if self.method == 'general':
            att_energies = self.general_score(hidden,encoder_output)
        elif self.method =='concat':
            att_energies = self.concat_score(hidden,encoder_output)
        elif self.method =='dot':
            att_energies = self.dot_score(hidden, encoder_output)

        # transpose max_lenght and batch_size dimension
        att_energies = att_energies.t()

        # return the softmax normalized probabilty scores (with added dimension)
        return F.softmax(att_energies,dim=1).unsqueeze(1)


In [ ]:
class LuongAttDecoder(nn.Module):
    def __init__(self,attn_model, embedding,hidden_size, output_size, n_layers=1, dropout=0.1):
        super().__init__()

        # keepfor reference 
        self.attn_model = attn_model
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.n_layers = n_layers
        self.dropout = dropout

        # define layers 
        self.embedding = embedding
        self.embedding_dropout = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden_size, hidden_size, n_layers, dropout=(0 if n_layers ==1 else dropout))
        self.concat = nn.Linear(hidden_size*2, hidden_size)
        self.out = nn.Linear(hidden_size, output_size)

        self.attn = Attn(attn_model, hidden_size)

    def forward(self, input_step, last_hidden, encoder_outputs):
        # Get embedding through uniderectional GRU
        embedded = self.embedding(input_step)
        embedded = self.embedding_dropout(embedded)
        # Forward through unidirectional GRU
        rnn_output, hidden = self.gru(embedded, last_hidden)
        # calculate attention weights from the current GRU output 
        attn_weights = self.attn(rnn_output, encoder_outputs)
        # multiply attention weights to encoder to get new "weight sum" context vector
        context = attn_weights.bmm(encoder_outputs.transpose(0,1))
        # concatenate weighted context vector and GRU output using Luong
        rnn_output = rnn_output.squeeze(0)
        context =context.squeeze(1)
        concat_input = torch.cat((rnn_output,context),1)
        concat_output = torch.tanh(self.concat(concat_input))
        # predict next word using Luong 
        output = self.out(concat_output)
        output = F.softmax(output, dim=1)
        # return output and final hidden state
        return output, hidden
    
         

# Masked loss


In [ ]:
def Maskloss(inp, target, mask,device):
    ntotal = mask.sum()
    crossentropy = -torch.log(torch.gather(inp,1,target.view(-1,1)).squeeze(1))
    loss = crossentropy.masked_select(mask).mean()
    loss = loss.to(device)
    return loss, ntotal.item()

# Seq to Seq 


In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self,encoder, decoder, device, ratio):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        self.teacher_forcing_ratio =ratio

    def forward(self, input_seq, input_length, target, mask, max_target_len, batch_size):
        loss =0
        printlosses=[]
        n_totals = 0
        encoder_ouputs, encoder_hidden = self.encoder(input_seq, input_length)
        decoder_hidden = encoder_hidden[:decoder.n_layers]
        decoder_input = torch.LongTensor([[SOS_token for _ in range(batch_size)]])
        decoder_input = decoder_input.to(self.device)
        use_teacher_forcing = True if random.random() < self.teacher_forcing_ratio else False

        if use_teacher_forcing:
            for t in range(max_target_len):
                decoder_output, decoder_hidden = self.decoder(decoder_input, decoder_hidden,encoder_ouputs)
                decoder_input = target[t].view(1,-1)
                decoder_input =decoder_input.to(self.device)
                mask_loss, nTotal = Maskloss(decoder_output, target[t], mask[t], self.device)
                loss += mask_loss
                printlosses.append(mask_loss.item()*nTotal)
                n_totals += nTotal

        else:
            for t in range(max_target_len):
                decoder_output, decoder_hidden =self.decoder(
                    decoder_input, decoder_hidden, encoder_ouputs
                )
                _, topi = decoder_output.topk(1)
                decoder_input = torch.LongTensor([[topi[i][0] for i in range(batch_size)]])
                decoder_input = decoder_input.to(self.device)
                mask_loss, nTotal =Maskloss(decoder_output, target[t], mask[t],self.device)
                loss += mask_loss
                printlosses.append(mask_loss.item()*nTotal)
                n_totals += nTotal
        return loss, n_totals, printlosses

# Training functions

In [ ]:
def train_step(input_varibale, lengths, target_variable,mask, seq2seq, seq2seq_optimizer, max_target_len, batch_size, clip, device):
    seq2seq_optimizer.zero_grad()
    input_varibale = input_varibale.to(device)
    target_variable= target_variable.to(device)
    mask = mask.to(device)
    lengths = lengths.to('cpu')
    loss, n_totals, printlosses = seq2seq(input_varibale, lengths, target_variable, mask,
                                          max_target_len,batch_size)
    loss.backward()
    # clip at its place
    _ = nn.utils.clip_grad_norm_(seq2seq.parameters(),clip)
    seq2seq_optimizer.step()

    return sum(printlosses)/ n_totals

def train(voc, pairs, seq2seq, seq2seq_optimizer, embedding, encoder_n_layers, decoder_n_layers, n_iteration,
          batch_size, print_every, clip, corpus_name, device):
    
    training_batches = [batch2traindata(voc, [random.choice(pairs) for _ in range(batch_size)])
                        for _ in range(n_iteration)]
    print("initializing ....")
    start_iteration = 1
    print_loss =0
    print("Training ....")
    for iteration in range(start_iteration, n_iteration+1):
        training_batch = training_batches[iteration -1]
        input_variable, lengths, target_variable, mask, max_target_len= training_batch
        loss = train_step(input_variable, lengths, target_variable, mask, seq2seq,
                          seq2seq_optimizer, max_target_len, batch_size, clip, device)
        print_loss += loss
        if iteration % print_every ==0:
            print_loss_avg = print_loss/ print_every
            print("Iteration: {}; Percent complete: {:.1f}%; Average loss: {:.4f}".format(iteration, iteration/n_iteration *100, print_loss_avg))
            print_loss =0

# Important Training Parameters

In [ ]:
attn_model = 'dot' #'general', 'concat' 
hidden_size =500
encoder_n_layers =2
decoder_n_layers =2
dropout =0.1
batch_size =64
clip =50
teacher_forcing_ratio =0.5
learning_rate =0.0005
n_iteration= 20000
print_every =500

# Initialize the model

In [ ]:
embedding =nn.Embedding(voc.num_words, hidden_size)
encoder = EncoderRNN(hidden_size, embedding, encoder_n_layers, dropout)
decoder = LuongAttDecoder(attn_model, embedding,hidden_size, voc.num_words, decoder_n_layers, dropout)
encoder =encoder.to(device)
decoder = decoder.to(device)
encoder.train()
decoder.train()
seq2seq = Seq2Seq(encoder,decoder, device, teacher_forcing_ratio)
seq2seq=seq2seq.to(device)
seq2seq.train()
seq2seq_optimizer = optim.Adam(seq2seq.parameters(), lr=learning_rate)

for state in seq2seq_optimizer.state.values():
    for k,v  in state.items():
        if isinstance(v,torch.Tensor):
            state[k] =v.cuda()

In [ ]:
corpus_name

'movie-corpus'

In [ ]:
train(voc, pairs, seq2seq, seq2seq_optimizer,
      embedding, encoder_n_layers, decoder_n_layers, n_iteration, batch_size, print_every, clip, corpus_name, device)

initializing ....
Training ....


KeyboardInterrupt: 

# Greedy decoding

In [ ]:
class GreedySearchDecoder(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, input_seq, input_length, max_length):
        # forward input through encoder model
        encoder_outputs , encoder_hidden = self.encoder(input_seq, input_length)
        # prepare encoder's final hidden layer to be first hidden input to the decoder
        decoder_hidden = encoder_hidden[:self.decoder.n_layers]
        # Initialize decoder input with SOS_token
        decoder_input = torch.ones(1,1,device=device, dtype=torch.long) *SOS_token
        # Initialize tensors to append  deocoded words to
        all_tokens = torch.zeros([0], device=device, dtype=torch.long)
        all_scores = torch.zeros([0], device=device)

        # Iteratively decode one word token at a time
        for _ in range(max_length):
            # Forward pass through decoder
            decoder_output ,decoder_hidden = self.decoder(decoder_input, decoder_hidden, encoder_outputs)
            # Obtain most likely word toekn and its softmax score
            decoder_scores , decoder_input = torch.max(decoder_output,dim=1)
            # record token and score 
            all_tokens = torch.cat((all_tokens,decoder_input), dim=0)
            all_scores = torch.cat((all_scores,decoder_scores),dim=0)
            # Prepare current token to be the next decoder input (add a dimension)
            decoder_input = torch.unsqueeze(decoder_input,0)

        # return collections if word tokens and scores
        return all_tokens, all_scores 

# Evaluate my text

In [ ]:
def evaluate(encoder, decoder, searcher, voc, sentence, max_length=MAX_LENGTH):
    ### Format input sentence as a batch
    # words -> indexes
    indexes_batch = [indexesfromsentence(voc, sentence)]

    # create lengths tensor
    lengths =  torch.tensor([len(indexes) for indexes in indexes_batch])
    # transpose dimensions of batch to match model's expectations
    input_batch = torch.LongTensor(indexes_batch).transpose(0,1)
    # Use appropriate device 
    input_batch = input_batch.to(device)
    lengths = lengths.to(device)
    # Decoder sentence with searcher
    tokens, scores =searcher(input_batch, lengths,max_length)
    # indexes -> words
    decoded_words = [voc.index2owrd[token.item()] for token in tokens]

    return decoded_words

def evaluateinput(encoder, decoder, searcher, voc):
    input_sentence = ''
    while(1):
        try:
            # get input sentence
            input_sentence =input(">>>")
            # check if it is quit case
            if input_sentence =='q' or input_sentence =='quit' :break
            # normalize sentence
            input_sentence = normalizestring(input_sentence)
            # evaluate  sentence 
            output_words = evaluate(encoder, decoder, searcher, voc, input_sentence)
            # format and print repsonse sentence
            output_words[:] = [x for x in output_words if not (x =='EOS' or x == 'PAD')] 
            print('Bot:', ''.join(output_words))

        except KeyError:
            print("Error: Encounterd unknown word.")


# Run model 

In [ ]:
# configure models
model_name = 'cb_model'
attn_model = 'dot'
# attn_model = 'geneeral'
# attn_model = 'concat' 
hidden_size = 500
encoder_n_layers = 2
decoder_n_layers =2
dropout = 0.1
batch_size = 64

# set checkpoint to load from; 
loadfilename = None
checkpoint_iter = 4000


In [ ]:

loadfilename = os.path.join('/media/mark/New Volume/data_science/convo_bot/real_model_1.pth')





In [ ]:
# load model if a '''loadfilename ''' is provided
if loadfilename:
    # if loading on same machine the model was trained on
    checkpoint = torch.load(loadfilename,map_location=torch.device('cpu'))
    #if loading a model trained on 
    encoder_sd = checkpoint['en']
    decoder_sd = checkpoint['de']
    encoder_optimizer_sd = checkpoint['en_opt']
    embedding_sd = checkpoint['embedding']
    voc.__dict__= checkpoint['voc_dict']


print('Building encoder and decoder ....')
# initialize 
embedding  = nn.Embedding(voc.num_words, hidden_size)
if loadfilename:
    embedding.load_state_dict(embedding_sd)
# initialize encoder & decoder models
encoder = EncoderRNN(hidden_size, embedding,encoder_n_layers, dropout)
decoder = LuongAttDecoder(attn_model, embedding, hidden_size, voc.num_words, decoder_n_layers, dropout)
if loadfilename:
    encoder.load_state_dict(encoder_sd)
    decoder.load_state_dict(decoder_sd)

# use appropriate device
encoder =encoder.to(device)
decoder = decoder.to(device)

print("Model built and ready  to go!")

Building encoder and decoder ....
Model built and ready  to go!


In [ ]:
# Set dropout layers to ``eval`` mode
encoder.eval()
decoder.eval()

# Initialize search module
searcher = GreedySearchDecoder(encoder, decoder)

# Begin chatting (uncomment and run the following line to begin)
# evaluateInput(encoder, decoder, searcher, voc)


In [58]:
evaluateinput(encoder,decoder,searcher,voc)

Bot: youare?.?
Bot: sincehowfind
Bot: youyou
Bot: areyouatmeaatoo?
Bot: okay?llthe
Bot: noimnot..
Bot: noinot..
Bot: becauseihavesomethingtheimportanttodo.
Bot: notso....
Bot: getoutofhere.
Bot: icanam.tyouyou.
